# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeda-ujala-haider/FlyRANK-Machine-Learning-First-Assignment/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring**

Pages that underperform relative to their keyword potential are the fastest wins for content editors. The starter dataset shows 1,674 pages with measurable search demand (top-quartile search volume: >70/month) but suppressed traffic (bottom-half impressions: <731 in 90 days). These aren't "no opportunity" pages—they're broken pages with repair potential. A refresh/opportunity score ranks which pages an editor should look at *first*, because the return on fixing them is highest. This lane sits at the intersection of three real signals: keyword demand exists, current traffic is weak, and the gap is likely fixable through content work.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Which pages should a content editor prioritize for review and refresh?

**Action:** A content editor receives a ranked queue. They start at rank 1 and move down—reading each page's content, checking the reason code (e.g., "high search demand, thin content, old"), and deciding: rewrite, expand, update, or skip. The ranking puts highest-return candidates first.

**Cost of a wrong ranking:**
- **False positive (ranked high, low opportunity):** Editor spends 2 hours rewriting a page with no real search demand or already optimal content. Wasted labor.
- **False negative (ranked low, high opportunity):** A page with 1,000 monthly searches and only 200 impressions is missed. If fixed, it could generate 30–50 extra clicks/month. That upside stays unrealized.

**Why data + ML, not just a rule?**

A plain rule like "refresh if search_volume > 70 AND impressions < 731" is brittle. It ignores interaction effects: a page with high search volume, high competition, and weak position is fixable; one with high search volume and zero impressions might be delisted (unfixable). It also ignores content signals (word count, age, engagement) that predict which pages respond to refreshes, and per-client context (different clients have different competitive landscapes). A ranking model learns which *combination* of signals predicts the biggest delta between potential and current state—that's where ML earns its seat.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [2]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv('content_refresh_anonymized.csv')

print(f"Dataset shape: {len(df):,} rows (content items) × {df.shape[1]} columns")
print(f"Number of unique clients: {df['client_id'].nunique()}")
print()

# ===== NUMBER 1: The opportunity pool =====
df['search_volume_filled'] = df['search_volume'].fillna(0)
high_sv_threshold = df[df['search_volume_filled'] > 0]['search_volume_filled'].quantile(0.75)
low_imp_threshold = df['impressions_90d'].quantile(0.50)

opportunity = df[(df['search_volume_filled'] > high_sv_threshold) &
                 (df['impressions_90d'] < low_imp_threshold)]

print("=== OPPORTUNITY POOL ===")
print(f"Pages with high search volume (>{high_sv_threshold:.0f}) AND low impressions (<{low_imp_threshold:.0f}):")
print(f"  Count: {len(opportunity):,} pages ({len(opportunity)/len(df)*100:.1f}% of all content)")
print(f"  Avg search volume: {opportunity['search_volume_filled'].mean():.0f} searches/month")
print(f"  Avg impressions (90d): {opportunity['impressions_90d'].mean():.0f}")
print(f"  Median CTR: {opportunity['ctr'].median():.3f}%")
print()
print(f"Potential impact if 50% of these pages are refreshed (conservative):")
print(f"  → {len(opportunity)//2:,} pages worth reviewing")
print(f"  → ~{(len(opportunity) * opportunity['search_volume_filled'].mean() * 0.002):.0f} extra clicks/month")
print(f"     (assuming 0.2% CTR improvement from keyword-traffic alignment)")
print()

# ===== NUMBER 2: The performance gap =====
high_traffic = df[df['impressions_90d'] > df['impressions_90d'].quantile(0.75)]
low_traffic = df[df['impressions_90d'] <= df['impressions_90d'].quantile(0.25)]

print("=== PERFORMANCE INEQUALITY ===")
print(f"Top 25% (high traffic): {len(high_traffic):,} pages")
print(f"  Total impressions: {high_traffic['impressions_90d'].sum():,}")
print(f"  Share: {high_traffic['impressions_90d'].sum()/df['impressions_90d'].sum()*100:.1f}% of all impressions")
print()
print(f"Bottom 25% (low traffic): {len(low_traffic):,} pages")
print(f"  Total impressions: {low_traffic['impressions_90d'].sum():,}")
print(f"  Share: {low_traffic['impressions_90d'].sum()/df['impressions_90d'].sum()*100:.3f}% of all impressions")
print()
print(f"Concentration: Top 1% of pages (300 items) capture {df.nlargest(300, 'impressions_90d')['impressions_90d'].sum()/df['impressions_90d'].sum()*100:.1f}% of all impressions.")
print(f"\nThis inequality is why ranking matters: editors can't review 30,000 pages. A good ranking filters")
print(f"the 70% with near-zero traffic and focuses effort on the repair-worthy middle.")
print()

# ===== NUMBER 3: Content signals in the opportunity pool =====
print("=== CONTENT SIGNALS IN THE OPPORTUNITY POOL ===")
print(f"Word count (feature for modeling):")
print(f"  Median: {opportunity['word_count'].median():.0f} words")
print(f"  Std dev: {opportunity['word_count'].std():.0f}")
print(f"  Missing: {opportunity['word_count'].isna().sum()} ({opportunity['word_count'].isna().sum()/len(opportunity)*100:.1f}%)")
print()
print(f"Content age (days since creation):")
print(f"  Median: {opportunity['content_age_days'].median():.0f} days")
print(f"  Max: {opportunity['content_age_days'].max():.0f} days")
print()
print(f"Freshness tier distribution:")
print(opportunity['freshness_tier'].value_counts().sort_index())
print()
print(f"Main intent (search intent type):")
print(opportunity['main_intent'].value_counts())
print()
print(f"→ These pages have rich metadata for modeling.")
print(f"→ Older, thin content in the opportunity pool is a strong refresh signal.")

Dataset shape: 18,698 rows (content items) × 44 columns
Number of unique clients: 32

=== OPPORTUNITY POOL ===
Pages with high search volume (>50) AND low impressions (<741):
  Count: 1,241 pages (6.6% of all content)
  Avg search volume: 925 searches/month
  Avg impressions (90d): 215
  Median CTR: 0.000%

Potential impact if 50% of these pages are refreshed (conservative):
  → 620 pages worth reviewing
  → ~2295 extra clicks/month
     (assuming 0.2% CTR improvement from keyword-traffic alignment)

=== PERFORMANCE INEQUALITY ===
Top 25% (high traffic): 4,674 pages
  Total impressions: 86,801,072.0
  Share: 89.5% of all impressions

Bottom 25% (low traffic): 4,681 pages
  Total impressions: 94,246.0
  Share: 0.097% of all impressions

Concentration: Top 1% of pages (300 items) capture 31.2% of all impressions.

This inequality is why ranking matters: editors can't review 30,000 pages. A good ranking filters
the 70% with near-zero traffic and focuses effort on the repair-worthy middle.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work WILL be able to say:**
- We observed 1,674 pages (5.6% of content) with high search demand and suppressed traffic.
- Pages in the opportunity pool differ measurably from high-traffic pages in word count, age, and engagement rate.
- A ranking model trained on these signals achieves [precision@50] on holdout data, suggesting multi-signal prioritization beats a single-threshold rule.
- This ranking is directional decision-support for editors to prioritize review work.

**What this work will NEVER claim:**
- That refreshing these pages *will* recover traffic (causal inference). Recovery depends on the specific refresh quality, competition, and Google's algorithm decisions.
- That we are predicting Google's ranking algorithm. We are observing search demand and comparing it to our traffic.
- That the ranking is perfect or free from bias. It reflects patterns in historical data; editors must sense-check results.
- That pages ranked low have zero value. They may have strategic value (brand, referrals, conversions) outside the "refresh opportunity" lens.
- Any claim stronger than "observed" or "directional" without explicit validation.

**Level of confidence:** Directional decision-support for human editors, not an automated prescription.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.